In [1]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

# Setup device
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

# Load the fine-tuned model and tokenizer
model_path = "./final_asl_gloss_model"
tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)

print(f"Model loaded onto {device}")

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Model loaded onto mps


In [2]:
# Load the original How2Sign metadata
df = pd.read_csv('how2sign_dataset/how2sign_realigned_test.csv', sep='\t')

# Preview to ensure it loaded correctly
print(df.head())

      VIDEO_ID               VIDEO_NAME    SENTENCE_ID  \
0  -fZc293MpJk  -fZc293MpJk-1-rgb_front  -fZc293MpJk_0   
1  -fZc293MpJk  -fZc293MpJk-1-rgb_front  -fZc293MpJk_2   
2  -fZc293MpJk  -fZc293MpJk-1-rgb_front  -fZc293MpJk_3   
3  -fZc293MpJk  -fZc293MpJk-1-rgb_front  -fZc293MpJk_4   
4  -fZc293MpJk  -fZc293MpJk-1-rgb_front  -fZc293MpJk_5   

               SENTENCE_NAME  START_REALIGNED  END_REALIGNED  \
0  -fZc293MpJk_0-1-rgb_front             0.26           6.79   
1  -fZc293MpJk_2-1-rgb_front             7.27          20.30   
2  -fZc293MpJk_3-1-rgb_front            21.25          25.51   
3  -fZc293MpJk_4-1-rgb_front            27.75          44.64   
4  -fZc293MpJk_5-1-rgb_front            46.68          52.44   

                                            SENTENCE  
0                                                Hi!  
1  The aileron is the control surface in the wing...  
2  By moving the stick, you cause pressure to inc...  
3  The elevator is the part that moves with th

In [3]:
import re
import torch
from tqdm.notebook import tqdm

# 1. Explicitly tell the tokenizer what language to expect and output
tokenizer.src_lang = "eng_Latn"
tokenizer.tgt_lang = "eng_Latn"

# Get the ID for the starting token
forced_bos_token_id = tokenizer.convert_tokens_to_ids("eng_Latn")

# CRITICAL FIX: Ensure the model is in full precision (FP32). 
# Remove any .half() calls to avoid the MPS NaN bug.
model.float().eval()

def generate_gloss(batch_sentences):
    # Preprocess the batch to match ASLG-PC12 training data
    cleaned_batch = []
    for sentence in batch_sentences:
        # Lowercase and remove punctuation
        clean_text = sentence.lower()
        clean_text = re.sub(r'[^\w\s]', '', clean_text)
        cleaned_batch.append(clean_text)

    # Tokenize the CLEANED sentences
    inputs = tokenizer(cleaned_batch, return_tensors="pt", padding=True, truncation=True).to(device)
    
    # Generate sequences WITH the attention mask and forced BOS token
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs["input_ids"], 
            attention_mask=inputs["attention_mask"], 
            max_length=128, 
            num_beams=2, 
            forced_bos_token_id=forced_bos_token_id, 
            early_stopping=True
        )
    
    # Decode back to text
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# Run the batching loop
batch_size = 32  # Keep at 32 or 64
generated_glosses = []

print("Starting full-precision inference...")
for i in tqdm(range(0, len(df), batch_size)):
    batch = df['SENTENCE'].iloc[i : i + batch_size].tolist()
    gloss_batch = generate_gloss(batch)
    generated_glosses.extend(gloss_batch)

# Add the new column to our dataframe
df['GENERATED_GLOSS'] = generated_glosses
print("Done! Check the output with: print(df[['SENTENCE', 'GENERATED_GLOSS']].head())")

Starting full-precision inference...


  0%|          | 0/74 [00:00<?, ?it/s]

Done! Check the output with: print(df[['SENTENCE', 'GENERATED_GLOSS']].head())


In [6]:
print(df[['SENTENCE', 'GENERATED_GLOSS']].head())

                                            SENTENCE  \
0                                                Hi!   
1  The aileron is the control surface in the wing...   
2  By moving the stick, you cause pressure to inc...   
3  The elevator is the part that moves with the s...   
4  Therefore, it's either going uphill, downhill,...   

                                     GENERATED_GLOSS  
0                                                 HI  
1  AILERON BE CONTROL SURFACE IN WING THAT BE CON...  
2  BY MOVE STICK X-YOU CAUSE PRESSURE TO INCREASE...  
3  ELEVATOR BE PART THAT MOVE WITH STICK DESC-FOR...  
4  DESC-REFORE X-ITS EIR GO DESC-UPHILL DOWNRHILL...  


In [7]:
output_filename = "how2sign_realigned_test_glosses.csv"
df.to_csv(output_filename, sep='\t', index=False)

print(f"Success! New dataset saved to {output_filename}")

Success! New dataset saved to how2sign_realigned_test_glosses.csv
